In [1]:
from pathlib import Path
import itertools
import json

import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve


# ============================================================
#
# Full nonlinear objective vs regression-only objective
# ============================================================

SEEDS = [42, 52, 62, 72, 82]
DURATIONS = [30, 60, 120]

FULL_ROOT = Path(
    r"D:\ONYEKA_NEW_EXPERIMENT\Onyeka_Review_Experiment\02_reliability_models"
)

REG_ROOT = Path(
    r"D:\ONYEKA_NEW_EXPERIMENT\reviewer8_loss_ablation\reg_only"
)

OUTPUT_ROOT = REG_ROOT / "reviewer8_analysis"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


# ============================================================
# Helpers
# ============================================================

def find_unique(root, filename):
    direct = root / filename

    if direct.is_file():
        return direct

    matches = list(root.rglob(filename))

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one file named {filename} under:\n"
            f"{root}\n"
            f"Found {len(matches)}:\n"
            + "\n".join(str(x) for x in matches)
        )

    return matches[0]


def interpolated_eer(y_true, scores):
    """
    Unknown users are the positive class.
    Larger scaled distance = more unknown-like.
    """

    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)

    fpr, tpr, _ = roc_curve(
        y_true,
        scores,
        pos_label=1,
    )

    fnr = 1.0 - tpr
    difference = fpr - fnr

    exact = np.where(
        np.isclose(
            difference,
            0.0,
            atol=1e-15,
        )
    )[0]

    if len(exact) > 0:
        i = int(exact[0])
        return float(
            0.5 * (fpr[i] + fnr[i])
        )

    crossing = np.where(
        difference > 0
    )[0]

    if len(crossing) == 0:
        i = int(
            np.argmin(
                np.abs(difference)
            )
        )

        return float(
            0.5 * (fpr[i] + fnr[i])
        )

    j = int(crossing[0])

    if j == 0:
        return float(
            0.5 * (fpr[0] + fnr[0])
        )

    i = j - 1

    d0 = float(difference[i])
    d1 = float(difference[j])

    if np.isclose(d1, d0):
        weight = 0.5
    else:
        weight = -d0 / (d1 - d0)

    fpr_cross = (
        fpr[i]
        + weight * (fpr[j] - fpr[i])
    )

    fnr_cross = (
        fnr[i]
        + weight * (fnr[j] - fnr[i])
    )

    return float(
        0.5 * (fpr_cross + fnr_cross)
    )


def exact_sign_flip_test(differences):
    """
    Two-sided exact sign-flip test over all 2^5 assignments.
    """

    differences = np.asarray(
        differences,
        dtype=float,
    )

    observed = abs(
        differences.mean()
    )

    null_values = []

    for signs in itertools.product(
        [-1.0, 1.0],
        repeat=len(differences),
    ):

        signs = np.asarray(
            signs,
            dtype=float,
        )

        null_values.append(
            abs(
                (
                    differences * signs
                ).mean()
            )
        )

    null_values = np.asarray(
        null_values
    )

    p_value = float(
        np.mean(
            null_values
            >=
            (observed - 1e-15)
        )
    )

    sd = differences.std(
        ddof=1
    )

    if np.isclose(sd, 0.0):
        dz = np.nan
    else:
        dz = float(
            differences.mean()
            / sd
        )

    return p_value, dz


def get_eer_from_predictions(
    predictions_path,
    duration,
):
    df = pd.read_csv(
        predictions_path
    )

    current = df.loc[
        pd.to_numeric(
            df["aggregation_seconds"]
        ).eq(duration)
    ].copy()

    labels = (
        current["known_or_unknown"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    y_unknown = (
        labels.eq("unknown")
        .astype(int)
        .to_numpy()
    )

    scores = (
        pd.to_numeric(
            current["scaled_distance"]
        )
        .to_numpy(dtype=float)
    )

    return interpolated_eer(
        y_unknown,
        scores,
    )


def extract_metrics_row(
    metrics_path,
    predictions_path,
    seed,
    duration,
):
    metrics = pd.read_csv(
        metrics_path
    )

    row = metrics.loc[
        pd.to_numeric(
            metrics["aggregation_seconds"]
        ).eq(duration)
    ]

    if len(row) != 1:
        raise RuntimeError(
            f"{metrics_path}\n"
            f"Expected exactly one row for {duration}s, "
            f"found {len(row)}."
        )

    row = row.iloc[0]

    known_acceptance_macro = float(
        row["known_acceptance_rate_macro_user"]
    )

    unknown_rejection_observed = float(
        row["unknown_true_rejection_rate"]
    )

    balanced_detection = (
        0.5
        * (
            known_acceptance_macro
            +
            unknown_rejection_observed
        )
    )

    eer = get_eer_from_predictions(
        predictions_path,
        duration,
    )

    return {
        "seed": seed,
        "aggregation_seconds": duration,
        "closed_accuracy":
            float(row["closed_set_accuracy"]),
        "auroc":
            float(row["unknown_detection_auroc"]),
        "aupr":
            float(row["unknown_detection_aupr"]),
        "interpolated_eer":
            eer,
        "oscr":
            float(row["oscr_auc"]),
        "balanced_detection":
            float(balanced_detection),
        "open_overall_accuracy":
            float(row["open_set_overall_accuracy"]),
        "known_acceptance_macro":
            known_acceptance_macro,
        "unknown_rejection_observed":
            unknown_rejection_observed,
    }


# ============================================================
# Verify reg-only configuration
# ============================================================

config_path = find_unique(
    REG_ROOT,
    "run_config.json",
)

with open(
    config_path,
    "r",
    encoding="utf-8",
) as f:
    config = json.load(f)


print("=" * 100)
print("REG-ONLY CONFIGURATION CHECK")
print("=" * 100)

for key in [
    "lambda_reliability",
    "lambda_gate_alignment",
    "lambda_monotonic",
    "epochs",
    "batch_size",
    "learning_rate",
    "corruption_probability",
]:
    print(
        f"{key}: {config.get(key)}"
    )


assert np.isclose(
    float(config["lambda_reliability"]),
    1.0,
)

assert np.isclose(
    float(config["lambda_gate_alignment"]),
    0.0,
)

assert np.isclose(
    float(config["lambda_monotonic"]),
    0.0,
)

print()
print("REG-ONLY CONFIGURATION: PASSED")


# ============================================================
# Read five-seed results
# ============================================================

rows = []

for seed in SEEDS:

    full_folder = (
        FULL_ROOT
        /
        f"protocol_seed{seed}_nonlinear"
    )

    full_metrics = (
        full_folder
        /
        f"seed_{seed}_metrics.csv"
    )

    full_predictions = (
        full_folder
        /
        f"seed_{seed}_predictions.csv"
    )

    reg_metrics = find_unique(
        REG_ROOT,
        f"seed_{seed}_metrics.csv",
    )

    reg_predictions = find_unique(
        REG_ROOT,
        f"seed_{seed}_predictions.csv",
    )

    print()
    print(f"Seed {seed}")
    print("  FULL metrics:", full_metrics)
    print("  REG  metrics:", reg_metrics)

    if not full_metrics.is_file():
        raise FileNotFoundError(
            full_metrics
        )

    if not full_predictions.is_file():
        raise FileNotFoundError(
            full_predictions
        )

    for duration in DURATIONS:

        full = extract_metrics_row(
            full_metrics,
            full_predictions,
            seed,
            duration,
        )

        full["method"] = "full"
        rows.append(full)

        reg = extract_metrics_row(
            reg_metrics,
            reg_predictions,
            seed,
            duration,
        )

        reg["method"] = "reg_only"
        rows.append(reg)


seedwise = pd.DataFrame(
    rows
)


# ============================================================
# Summary mean ± SD
# ============================================================

METRICS = [
    "closed_accuracy",
    "auroc",
    "aupr",
    "interpolated_eer",
    "oscr",
    "balanced_detection",
    "open_overall_accuracy",
]


summary_rows = []

for method in [
    "full",
    "reg_only",
]:

    for duration in DURATIONS:

        current = seedwise.loc[
            seedwise["method"].eq(method)
            &
            seedwise[
                "aggregation_seconds"
            ].eq(duration)
        ]

        result = {
            "method": method,
            "aggregation_seconds": duration,
        }

        for metric in METRICS:

            result[
                f"{metric}_mean"
            ] = float(
                current[metric].mean()
            )

            result[
                f"{metric}_sd"
            ] = float(
                current[metric].std(
                    ddof=1
                )
            )

        summary_rows.append(
            result
        )


summary = pd.DataFrame(
    summary_rows
)


# ============================================================
# Validate reconstruction of existing full nonlinear results
# ============================================================

EXPECTED_FULL = {
    30: {
        "closed_accuracy": 0.3196,
        "auroc": 0.6644,
        "aupr": 0.4227,
        "interpolated_eer": 0.3845,
        "oscr": 0.2558,
        "balanced_detection": 0.5458,
        "open_overall_accuracy": 0.3391,
    },
    60: {
        "closed_accuracy": 0.3255,
        "auroc": 0.6672,
        "aupr": 0.4296,
        "interpolated_eer": 0.3819,
        "oscr": 0.2626,
        "balanced_detection": 0.5369,
        "open_overall_accuracy": 0.3243,
    },
    120: {
        "closed_accuracy": 0.3257,
        "auroc": 0.6701,
        "aupr": 0.4298,
        "interpolated_eer": 0.3788,
        "oscr": 0.2656,
        "balanced_detection": 0.5629,
        "open_overall_accuracy": 0.3617,
    },
}


print()
print("=" * 100)
print("FULL NONLINEAR VALIDATION")
print("=" * 100)

for duration in DURATIONS:

    row = summary.loc[
        summary["method"].eq("full")
        &
        summary[
            "aggregation_seconds"
        ].eq(duration)
    ].iloc[0]

    print()
    print(f"{duration} s")

    for metric, expected in EXPECTED_FULL[
        duration
    ].items():

        actual = float(
            row[f"{metric}_mean"]
        )

        difference = (
            actual - expected
        )

        print(
            f"  {metric:28s} "
            f"actual={actual:.6f} "
            f"paper={expected:.4f} "
            f"diff={difference:+.6f}"
        )

        if abs(difference) > 0.00055:
            raise RuntimeError(
                f"Validation failed for "
                f"{duration}s {metric}"
            )

print()
print("FULL NONLINEAR VALIDATION: PASSED")


# ============================================================
# Paired exact sign-flip tests
#
# Positive full advantage = full objective performs better.
#
# Higher-is-better:
#     full - reg_only
#
# EER:
#     reg_only - full
# ============================================================

paired_rows = []

for duration in DURATIONS:

    full = (
        seedwise.loc[
            seedwise["method"].eq("full")
            &
            seedwise[
                "aggregation_seconds"
            ].eq(duration)
        ]
        .sort_values("seed")
        .reset_index(drop=True)
    )

    reg = (
        seedwise.loc[
            seedwise["method"].eq("reg_only")
            &
            seedwise[
                "aggregation_seconds"
            ].eq(duration)
        ]
        .sort_values("seed")
        .reset_index(drop=True)
    )

    if not np.array_equal(
        full["seed"].to_numpy(),
        reg["seed"].to_numpy(),
    ):
        raise RuntimeError(
            f"Seed alignment failed at {duration}s"
        )

    for metric in METRICS:

        if metric == "interpolated_eer":

            differences = (
                reg[metric].to_numpy(
                    dtype=float
                )
                -
                full[metric].to_numpy(
                    dtype=float
                )
            )

        else:

            differences = (
                full[metric].to_numpy(
                    dtype=float
                )
                -
                reg[metric].to_numpy(
                    dtype=float
                )
            )

        p_value, dz = exact_sign_flip_test(
            differences
        )

        paired_rows.append({
            "aggregation_seconds":
                duration,
            "metric":
                metric,
            "mean_full_advantage":
                float(
                    differences.mean()
                ),
            "sd_full_advantage":
                float(
                    differences.std(
                        ddof=1
                    )
                ),
            "full_better_seeds":
                int(
                    (
                        differences > 0
                    ).sum()
                ),
            "reg_only_better_seeds":
                int(
                    (
                        differences < 0
                    ).sum()
                ),
            "ties":
                int(
                    np.isclose(
                        differences,
                        0.0
                    ).sum()
                ),
            "cohen_dz":
                dz,
            "exact_sign_flip_p":
                p_value,
        })


paired = pd.DataFrame(
    paired_rows
)


# ============================================================
# Print the results you need to send me
# ============================================================

print()
print("=" * 100)
print("FULL VS REG-ONLY")
print("=" * 100)

for duration in DURATIONS:

    print()
    print(f"{duration} s")

    for method in [
        "full",
        "reg_only",
    ]:

        row = summary.loc[
            summary["method"].eq(method)
            &
            summary[
                "aggregation_seconds"
            ].eq(duration)
        ].iloc[0]

        print(
            f"{method:8s} | "
            f"Closed "
            f"{row['closed_accuracy_mean']:.6f}"
            f" ± {row['closed_accuracy_sd']:.6f} | "
            f"AUROC "
            f"{row['auroc_mean']:.6f}"
            f" ± {row['auroc_sd']:.6f} | "
            f"AUPR "
            f"{row['aupr_mean']:.6f}"
            f" ± {row['aupr_sd']:.6f} | "
            f"EER "
            f"{row['interpolated_eer_mean']:.6f}"
            f" ± {row['interpolated_eer_sd']:.6f} | "
            f"OSCR "
            f"{row['oscr_mean']:.6f}"
            f" ± {row['oscr_sd']:.6f} | "
            f"Bdet "
            f"{row['balanced_detection_mean']:.6f}"
            f" ± {row['balanced_detection_sd']:.6f} | "
            f"Open "
            f"{row['open_overall_accuracy_mean']:.6f}"
            f" ± {row['open_overall_accuracy_sd']:.6f}"
        )


print()
print("=" * 100)
print("PAIRED FULL-OBJECTIVE ADVANTAGE")
print("Positive delta means the full three-loss objective performed better.")
print("=" * 100)

for duration in DURATIONS:

    print()
    print(f"{duration} s")

    current = paired.loc[
        paired[
            "aggregation_seconds"
        ].eq(duration)
    ]

    for _, row in current.iterrows():

        print(
            f"{row['metric']:28s} | "
            f"delta="
            f"{row['mean_full_advantage']:+.6f} | "
            f"full better="
            f"{int(row['full_better_seeds'])}/5 | "
            f"dz="
            f"{row['cohen_dz']:.3f} | "
            f"p="
            f"{row['exact_sign_flip_p']:.4f}"
        )


# ============================================================
# Save CSV outputs
# ============================================================

seedwise_path = (
    OUTPUT_ROOT
    /
    "reviewer8_full_vs_reg_only_seedwise.csv"
)

summary_path = (
    OUTPUT_ROOT
    /
    "reviewer8_full_vs_reg_only_summary.csv"
)

paired_path = (
    OUTPUT_ROOT
    /
    "reviewer8_full_vs_reg_only_paired_stats.csv"
)


seedwise.to_csv(
    seedwise_path,
    index=False,
)

summary.to_csv(
    summary_path,
    index=False,
)

paired.to_csv(
    paired_path,
    index=False,
)


print()
print("=" * 100)
print("FILES SAVED")
print("=" * 100)
print(summary_path)
print(paired_path)
print(seedwise_path)

print()
print("REVIEWER #8 ANALYSIS COMPLETED SUCCESSFULLY")

REG-ONLY CONFIGURATION CHECK
lambda_reliability: 1.0
lambda_gate_alignment: 0.0
lambda_monotonic: 0.0
epochs: 30
batch_size: 64
learning_rate: 0.001
corruption_probability: 0.6

REG-ONLY CONFIGURATION: PASSED

Seed 42
  FULL metrics: D:\ONYEKA_NEW_EXPERIMENT\Onyeka_Review_Experiment\02_reliability_models\protocol_seed42_nonlinear\seed_42_metrics.csv
  REG  metrics: D:\ONYEKA_NEW_EXPERIMENT\reviewer8_loss_ablation\reg_only\seed_42_metrics.csv

Seed 52
  FULL metrics: D:\ONYEKA_NEW_EXPERIMENT\Onyeka_Review_Experiment\02_reliability_models\protocol_seed52_nonlinear\seed_52_metrics.csv
  REG  metrics: D:\ONYEKA_NEW_EXPERIMENT\reviewer8_loss_ablation\reg_only\seed_52_metrics.csv

Seed 62
  FULL metrics: D:\ONYEKA_NEW_EXPERIMENT\Onyeka_Review_Experiment\02_reliability_models\protocol_seed62_nonlinear\seed_62_metrics.csv
  REG  metrics: D:\ONYEKA_NEW_EXPERIMENT\reviewer8_loss_ablation\reg_only\seed_62_metrics.csv

Seed 72
  FULL metrics: D:\ONYEKA_NEW_EXPERIMENT\Onyeka_Review_Experiment\02_re